# Frequency Law v9.0
## From Spin to the Universe — Combined & Complete Edition

**Author:** Christian Berrang  
**DOI:** 10.5281/zenodo.17874830

> *"The equations stay the same. The direction of reading changes."*

---

| Section | Content |
|---|---|
| 1 | Physical constants |
| 2 | Axioms A0-A6 (machine-readable) |
| 3 | Core formulas |
| 4 | Causal direction: f -> m (the key point) |
| 5 | Particle class & database |
| 6 | Validation against PDG (5 particles) |
| 7 | Pauli Principle as geometry |
| 8 | Mobius topology & phase cycles |
| 9 | Predictions: Berrangium O & Stoecker S |
| 10 | Frequency Periodic Table |
| 11 | Experimental test matrix |
| 12 | JSON framework export |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
from dataclasses import dataclass
from matplotlib.patches import Patch

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

h = 6.62607015e-34
c = 299792458
c2 = c**2
eV = 1.602176634e-19
G = 6.6743e-11
k_B = 1.380649e-23
hbar = h / (2 * np.pi)

print(f'Python: {sys.version.split()[0]}')
print(f'h = {h:.10e} J*s')
print(f'c = {c} m/s')
print(f'hbar = {hbar:.10e} J*s')
print('Constants loaded OK')

## 2. Axioms A0-A6

| ID | Name | Formal | Status |
|---|---|---|---|
| A0 | Null Field | N := {dPhi=0} | definition |
| A1 | Frequency is primary | f [Hz] | primary |
| A2 | Information | I ~ dPhi | primary |
| A3 | Time is emergent | T = dPhi / f | derived |
| A4 | Energy is derived | E = h*f | derived |
| A5 | Mass = bound frequency | m = h*f / c^2 | derived |
| A6 | Frequency conservation | sum h*fi = const | hypothesis |

In [ ]:
axioms = [
    {'id':'A0','name':'Null Field',             'formal':'N := {dPhi=0}',  'status':'definition'},
    {'id':'A1','name':'Frequency is primary',   'formal':'f [Hz]',         'status':'primary'},
    {'id':'A2','name':'Information',            'formal':'I ~ dPhi',       'status':'primary'},
    {'id':'A3','name':'Time is emergent',       'formal':'T = dPhi / f',   'status':'derived'},
    {'id':'A4','name':'Energy is derived',      'formal':'E = h*f',        'status':'derived'},
    {'id':'A5','name':'Mass = bound frequency', 'formal':'m = h*f / c^2',  'status':'derived'},
    {'id':'A6','name':'Freq. conservation',     'formal':'sum h*fi = const','status':'hypothesis'}
]
display(pd.DataFrame(axioms))

## 3. Core Formulas

In [ ]:
def mass_from_frequency(f_hz):
    return (h * f_hz) / c2

def frequency_from_mass(mass_kg):
    return (mass_kg * c2) / h

def MeV_to_kg(mass_MeV):
    return (mass_MeV * 1e6 * eV) / c2

def zitterbewegung_frequency(f_compton, topology='Mobius'):
    return 2 * f_compton if topology == 'Mobius' else f_compton

print('Core formulas loaded OK')

## 4. Causal Direction: f -> m

| Framework | Causal direction | Primary |
|---|---|---|
| Frequency Law | f -> m | frequency |
| Standard Model | m -> f | mass |

PDG values below are back-calculated for validation only. Causality always runs f -> m.

## 5. Particle Class & Database

In [ ]:
@dataclass
class Particle:
    name: str
    mass_MeV: float
    topology: str
    status: str
    generation: int = None

    @property
    def mass_kg(self):
        return MeV_to_kg(self.mass_MeV)

    @property
    def compton_freq(self):
        return frequency_from_mass(self.mass_kg)

    @property
    def zitter_freq(self):
        top = 'Mobius' if 'Mobius' in self.topology else 'circle'
        return zitterbewegung_frequency(self.compton_freq, top)

FERMIONS = [
    Particle('Neutrino v1',         0.000002, 'Mobius 4pi', 'known',      1),
    Particle('Electron e-',         0.511,    'Mobius 4pi', 'known',      1),
    Particle('Berrangium Omega',    16.2,     'Mobius 4pi', 'PREDICTION'   ),
    Particle('Muon mu-',           105.7,    'Mobius 4pi', 'known',      2),
    Particle('Stoecker Particle',  530.0,    'Mobius 4pi', 'PREDICTION'   ),
    Particle('Proton p',           938.3,    'Mobius 4pi', 'known',      1),
    Particle('Tau tau-',          1777.0,    'Mobius 4pi', 'known',      3),
    Particle('Bottom quark b',    4180.0,    'Mobius 4pi', 'known',      3),
    Particle('Top quark t',     172700.0,    'Mobius 4pi', 'known',      3),
]
BOSONS = [
    Particle('W Boson',   80400.0, 'circle 2pi', 'known'),
    Particle('Higgs H',  125100.0, 'circle 2pi', 'known'),
]
ALL_PARTICLES = FERMIONS + BOSONS
print(f'Loaded {len(FERMIONS)} fermions, {len(BOSONS)} bosons')
print(f'{sum(1 for p in ALL_PARTICLES if p.status=="PREDICTION")} predictions included')

## 6. Validation Against PDG

In [ ]:
PDG = {
    'Electron': {'mass_pdg_kg': 9.1093837015e-31,  'f_model_hz': 1.2358e20},
    'Proton':   {'mass_pdg_kg': 1.67262192369e-27,  'f_model_hz': 2.2687e23},
    'Neutron':  {'mass_pdg_kg': 1.67492749804e-27,  'f_model_hz': 2.2718e23},
    'Muon':     {'mass_pdg_kg': 1.88353162e-28,     'f_model_hz': 2.5554e22},
    'Higgs':    {'mass_pdg_kg': 2.225e-25,          'f_model_hz': 3.018e25},
}
rows = []
for name, vals in PDG.items():
    m_pdg  = vals['mass_pdg_kg']
    f_mod  = vals['f_model_hz']
    m_calc = mass_from_frequency(f_mod)
    dev    = abs(m_calc - m_pdg) / m_pdg * 100
    rows.append({'Particle': name, 'f_model (Hz)': f_mod,
                 'm_calc (kg)': m_calc, 'm_PDG (kg)': m_pdg,
                 'Deviation %': round(dev, 6)})
df_valid = pd.DataFrame(rows)
display(df_valid)
print('Small deviations = fingerprint of f -> m causality')

In [ ]:
colors = ['#2ecc71' if x < 0.01 else '#e67e22' for x in df_valid['Deviation %']]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(df_valid['Particle'], df_valid['Deviation %'], color=colors, edgecolor='white')
ax.set_ylabel('Deviation [%] log scale')
ax.set_title('Validation: m = h*f / c^2 vs PDG — no free parameters')
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.4)
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, df_valid['Deviation %']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.15,
            f'{val:.4f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Pauli Principle as Geometry

For identical fermions: Psi_total = psi1 - psi2 = 0. Not a rule. Topologically enforced.

In [ ]:
def pauli_wavefunction(psi1, psi2):
    return psi1 + psi2 * np.exp(1j * np.pi)

Psi_id   = pauli_wavefunction(1.0+0j, 1.0+0j)
Psi_diff = pauli_wavefunction(1.0+0j, 0.5+0.5j)

print(f'Identical fermions:  Psi = {Psi_id}  |Psi|^2 = {abs(Psi_id)**2} -> Forbidden')
print(f'Different fermions:  Psi = {Psi_diff:.4f}  |Psi|^2 = {abs(Psi_diff)**2:.4f} -> Allowed')

## 8. Mobius Topology & Phase Cycles

In [ ]:
print('Fermions (Mobius — spin-1/2):')
print(f'  dPhi = 4pi = {4*np.pi:.6f} rad')
print(f'  After 2pi: psi -> -psi (sign flip)')
print(f'  After 4pi: psi -> +psi (return)')
print(f'  -> f_zitter = 2 x f_compton (from topology, not fitted)')
print()
print('Bosons (circle — integer spin):')
print(f'  dPhi = 2pi = {2*np.pi:.6f} rad')
print(f'  After 2pi: psi -> +psi (return)')

## 9. Predictions: Berrangium & Stoecker

In [ ]:
b = next(p for p in FERMIONS if 'Berrangium' in p.name)
s = next(p for p in FERMIONS if 'Stoecker' in p.name)

print('BERRANGIUM OMEGA')
print(f'  Mass:              {b.mass_MeV} MeV/c^2')
print(f'  Compton freq:      {b.compton_freq:.4e} Hz')
print(f'  Position:          Between electron and muon')
print(f'  Hint:              X17 anomaly (Atomki) at ~17 MeV')
print(f'  Status:            Open')
print()
print('STOECKER PARTICLE')
print(f'  Mass:              {s.mass_MeV} MeV/c^2')
print(f'  Compton freq:      {s.compton_freq:.4e} Hz')
print(f'  Position:          Between muon and proton')
print(f'  Hint:              f0(500) resonance at 400-550 MeV')
print(f'  Dedication:        Prof. Dr. Horst Stoecker, FIAS Frankfurt')
print(f'  Status:            Open')

## 10. Frequency Periodic Table

In [ ]:
pp     = sorted([p for p in ALL_PARTICLES if p.mass_MeV > 0], key=lambda p: p.compton_freq)
names  = [p.name for p in pp]
freqs  = [p.compton_freq for p in pp]
colors = ['#e74c3c' if p.status=='PREDICTION'
          else '#3498db' if 'Mobius' in p.topology
          else '#2ecc71' for p in pp]

fig, ax = plt.subplots(figsize=(14, 9))
ax.barh(range(len(names)), freqs, color=colors, alpha=0.85, edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Compton Frequency (Hz)', fontsize=12, fontweight='bold')
ax.set_title('Frequency Periodic Table', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, axis='x')
ax.legend(handles=[
    Patch(facecolor='#3498db', alpha=0.85, label='Known fermions'),
    Patch(facecolor='#2ecc71', alpha=0.85, label='Known bosons'),
    Patch(facecolor='#e74c3c', alpha=0.85, label='Predictions')
], loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

## 11. Experimental Test Matrix

| Priority | Test | Method | Status |
|---|---|---|---|
| High | Phase-time T = dPhi/f | Mach-Zehnder | Open |
| High | Berrangium at ~16.2 MeV | Accelerator 15-17 MeV | Open |
| High | Stoecker at ~530 MeV | LHCb, BESIII, GlueX | Open |
| Medium | Time dilation T = dPhi/f | GPS, atomic clocks | Compatible |
| Exploratory | Zitterbewegung 2x | Electron scattering | Consistent |

All predictions are falsifiable. The gaps are either there or they are not.

## 12. JSON Framework Export

In [ ]:
framework = {
    'framework': 'Frequency Law',
    'version': '9.0',
    'doi': '10.5281/zenodo.17874830',
    'causal_direction': 'f -> dPhi -> T -> m -> E',
    'axioms': {
        'A0': {'name':'Null Field',            'formal':'N := {dPhi=0}', 'status':'definition'},
        'A1': {'name':'Frequency primary',      'formal':'f [Hz]',        'status':'primary'},
        'A2': {'name':'Phase = information',    'formal':'I ~ dPhi',       'status':'primary'},
        'A3': {'name':'Time emergent',          'formal':'T = dPhi / f',  'status':'derived'},
        'A4': {'name':'Energy derived',         'formal':'E = h*f',       'status':'derived'},
        'A5': {'name':'Mass = bound frequency', 'formal':'m = h*f / c^2', 'status':'derived'},
        'A6': {'name':'Freq. conservation',     'formal':'sum h*fi = const','status':'hypothesis'}
    },
    'falsification_targets': [
        {'name':'Berrangium Omega', 'predicted_MeV': 16.2,  'search_range_MeV':[15,17],   'status':'open'},
        {'name':'Stoecker Particle','predicted_MeV': 530.0, 'search_range_MeV':[450,600], 'status':'open'},
        {'name':'Phase-time',       'formula':'T = dPhi/f', 'method':'Mach-Zehnder',      'status':'open'}
    ]
}
print(json.dumps(framework, indent=2))

## Summary

| Result | Value |
|---|---|
| Electron Compton frequency | 1.2356 x 10^20 Hz (deviation < 0.02%) |
| Zitterbewegung factor | exactly 2x from Mobius topology |
| Pauli Principle | geometrically enforced, Psi_total = 0 |
| Causal direction | f -> m (not m -> f) |
| Berrangium | ~16.2 MeV — search open |
| Stoecker | ~530 MeV — search open |

---

Every particle is a clock. Every clock has its own frequency. Every frequency has its own Mobius loop.

**Frequency Law v9.0** | Christian Berrang | DOI: 10.5281/zenodo.17874830